<a href="https://colab.research.google.com/github/Kwasi-Scientist/Algorithms/blob/main/pbmc3k_AnnSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# AnnSQL + Scanpy stack (Colab)
!pip -q install annsql scanpy anndata pandas matplotlib duckdb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.3/263.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 54.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scanpy as sc
from AnnSQL import AnnSQL
from AnnSQL.MakeDb import MakeDb

# Configure DuckDB for memory and performance
db_config = {
    "memory_limit": "12GB", # Increase memory limit to 8GB
    "preserve_insertion_order": False, # Disable insertion order preservation to reduce memory
    "threads": 1 # Reduce number of threads to reduce memory
}

# Build an on-disk AnnSQL database (Colab-friendly)
adata = sc.datasets.pbmc3k()
# Add delete_existing_db=True to overwrite if it exists and reduce chunk_size for memory management
# Also set make_buffer_file=True to avoid loading all data into memory at once during database creation
MakeDb(adata=adata, db_name="pbmc3k", db_path="db/", db_config=db_config, delete_existing_db=True, chunk_size=300, make_buffer_file=True)  # creates db/pbmc3k.asql


# Open the materialized DB
asql = AnnSQL(db="db/pbmc3k.asql")




print("AnnSQL ready.")

  0%|          | 0.00/5.58M [00:00<?, ?B/s]

Time to make var_names unique:  46.51905012130737
Time to create X table structure:  0.4395163059234619
Starting chunked mode X table data insert. Total rows: 2700
Processed chunk 0-299 in 5.897182464599609 seconds
Processed chunk 300-599 in 4.498551607131958 seconds
Processed chunk 600-899 in 5.443653345108032 seconds
Processed chunk 900-1199 in 4.426806449890137 seconds
Processed chunk 1200-1499 in 4.4057300090789795 seconds
Processed chunk 1500-1799 in 5.267366647720337 seconds
Processed chunk 1800-2099 in 4.313692092895508 seconds
Processed chunk 2100-2399 in 5.109511375427246 seconds
Processed chunk 2400-2699 in 4.380814075469971 seconds

Too close for missiles, switching to guns
Creating X table from buffer file.
This may take a while...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Time to create X table from buffer: 582.4399363994598
Finished inserting X data.
Finished inserting obs data
Finished inserting var_names data
Finished inserting var data
Finished inserting obsm data
Finished inserting varm data
Finished inserting obsp data
AnnSQL ready.


In [3]:
def safe_gene_identifier(gene: str) -> str:
    if not isinstance(gene, str) or not gene:
        raise ValueError("Gene must be a non-empty string.")
    if gene.replace("_","").isalnum():
        return gene
    return f'"{gene}"'  # quote unusual symbols

def sql_total_counts_from_obs():
    # This function is not used after the fix, as total_counts are directly computed.
    # However, keeping it for context.
    return "SELECT cell_id, total_counts FROM obs ORDER BY total_counts DESC"

def sql_marker_positive_cells(marker: str):
    m = safe_gene_identifier(marker)
    return f"SELECT cell_id, {m} AS expr FROM X WHERE {m} > 0 ORDER BY expr DESC"

def detected_genes_per_cell_python(asql):
    # Pull a header to learn gene columns
    x_head = asql.query("SELECT * FROM X LIMIT 1")
    gene_cols = [c for c in x_head.columns if c != "cell_id"]

    # Bring wide X into pandas and compute >0 count per cell
    X_df = asql.query(f"SELECT cell_id, {', '.join([safe_gene_identifier(g) for g in gene_cols])} FROM X")
    detected = (X_df[[c for c in X_df.columns if c != "cell_id"]] > 0).sum(axis=1)
    df_detected = pd.DataFrame({"cell_id": X_df["cell_id"], "detected_genes": detected})

    # Calculate total counts using SQL query (replaces asql.calculate_total_counts())
    sum_expr_sql = ' + '.join([safe_gene_identifier(g) for g in gene_cols])
    counts = asql.query(f"SELECT cell_id, {sum_expr_sql} AS total_counts FROM X")

    return df_detected.merge(counts, on="cell_id", how="left")

In [5]:
MARKER = "MS4A1"  # e.g., B-cell marker; change to CD3D, NKG7, LYZ, etc.

# (1) Total UMI counts per cell
asql.calculate_total_counts()  # populates obs.total_counts
df_counts = asql.query(sql_total_counts_from_obs())
df_counts.head(3)

# (2) Marker-positive cells (wide X table with gene columns)
df_marker = asql.query(sql_marker_positive_cells(MARKER))
df_marker.head(3)


# (3) Gene-level summary - FIX FOR NONEType ERROR in asql.calculate_gene_counts()
x_head = asql.query("SELECT * FROM X LIMIT 1")
gene_cols = [c for c in x_head.columns if c != "cell_id"]

if not gene_cols:
    gene_counts = pd.DataFrame(columns=["gene", "sum", "nonzero_count"])
else:
    # Build a list of expressions for SUM and NONZERO_COUNT for each gene
    sum_expressions = []
    nonzero_count_expressions = []
    for gene in gene_cols:
        safe_g = safe_gene_identifier(gene)
        sum_expressions.append(f"SUM({safe_g}) AS {safe_g}_sum")
        nonzero_count_expressions.append(f"SUM(CASE WHEN {safe_g} > 0 THEN 1 ELSE 0 END) AS {safe_g}_nonzero_count")

    # Construct the SQL queries
    sum_query = f"SELECT {', '.join(sum_expressions)} FROM X"
    nonzero_query = f"SELECT {', '.join(nonzero_count_expressions)} FROM X"

    # Execute queries
    gene_sums_df = asql.query(sum_query)
    gene_nonzeros_df = asql.query(nonzero_query)

    # Reshape the results into a single DataFrame
    # Both gene_sums_df and gene_nonzeros_df will have a single row.
    gene_sums_melted = gene_sums_df.T.reset_index()
    gene_sums_melted.columns = ["gene_col", "sum"]
    gene_sums_melted["gene"] = gene_sums_melted["gene_col"].apply(lambda x: x.replace('_sum', '').replace('"', ''))

    gene_nonzeros_melted = gene_nonzeros_df.T.reset_index()
    gene_nonzeros_melted.columns = ["gene_col", "nonzero_count"]
    gene_nonzeros_melted["gene"] = gene_nonzeros_melted["gene_col"].apply(lambda x: x.replace('_nonzero_count', '').replace('"', ''))

    # Merge them
    gene_counts = pd.merge(gene_sums_melted[["gene", "sum"]], gene_nonzeros_melted[["gene", "nonzero_count"]], on="gene")

# The rest of the original logic remains, now operating on the robust `gene_counts` DataFrame.
n_cells = len(df_counts)
if "sum" in gene_counts.columns:
    gene_counts["mean_expression"] = gene_counts["sum"] / max(n_cells, 1)

top_cells = gene_counts.sort_values("nonzero_count", ascending=False).head(25) if "nonzero_count" in gene_counts.columns else None
top_mean  = gene_counts.sort_values("mean_expression",  ascending=False).head(25) if "mean_expression"  in gene_counts.columns else None

gene_counts.head(3)



# (4) Detected genes vs total counts (Python-first operation on wide X)
df_detected = detected_genes_per_cell_python(asql)
df_detected.head(3)

Query Successful
Total Counts Calculation Started


KeyboardInterrupt: 

In [ ]:
df_counts.to_csv("total_counts_per_cell.csv", index=False)
df_marker.to_csv(f"marker_{MARKER}_cells.csv", index=False)
if top_cells is not None:
    top_cells.to_csv("genes_detected_in_most_cells.csv", index=False)
if top_mean is not None:
    top_mean.to_csv("top_mean_expression_genes.csv", index=False)
df_detected.to_csv("counts_vs_detected_genes.csv", index=False)

print("CSVs written to the Colab working directory.")


In [ ]:
# 1) Histogram of total UMIs per cell
plt.figure()
plt.hist(df_counts["total_counts"], bins=50)
plt.title("Total UMI counts per cell (AnnSQL)")
plt.xlabel("UMIs")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# 2) Scatter: detected genes vs total counts
plt.figure()
plt.scatter(df_detected["total_counts"], df_detected["detected_genes"], s=6, alpha=0.6)
plt.title("Detected genes vs total counts (AnnSQL/Python)")
plt.xlabel("Total UMIs per cell")
plt.ylabel("Detected genes per cell")
plt.tight_layout()
plt.show()


In [ ]:
# 3) Bar: top mean-expression genes (if computed)
if top_mean is not None and "gene" in top_mean.columns and "mean_expression" in top_mean.columns:
    plt.figure(figsize=(10, 4))
    plt.bar(top_mean["gene"], top_mean["mean_expression"])
    plt.title("Top mean-expression genes (AnnSQL)")
    plt.xlabel("Gene")
    plt.ylabel("Mean expression")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/annsql_pbmc3k_outputs"
!cp -v *.csv "/content/drive/MyDrive/annsql_pbmc3k_outputs/" 2>/dev/null || true
print("Copied CSVs to Drive. Download figures by right-clicking on them above or saving via plt.savefig().")
